# Transformer Architectures & Downstream NLP Heads (BERT)

This interactive lab demonstrates how a Transformer encoder (BERT) powers core NLP tasks:
1. **Mini-BERT Backbone**: Embeddings (token, position, segment) and multi-head self-attention.
2. **Sentence Classification**: Sentiment analysis using the pooled `[CLS]` token.
3. **Token Classification (NER)**: Named Entity Recognition predicting per-token BIO tags.
4. **Extractive Question Answering**: Span prediction (start & end boundary logits) on context passages.
5. **End-to-End Optimization**: Backpropagation through task heads and transformer layers.

In [ ]:
import sys
from pathlib import Path
for p in [Path.cwd() / 'code', Path('03-nlp/transformer-models/code')]:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))

import torch
import torch.nn.functional as F
from transformer_tasks import (
    MiniBertModel,
    BertForSequenceClassification,
    BertForTokenClassification,
    BertForQuestionAnswering,
    extract_best_answer_span
)

print('Transformer & Downstream NLP heads loaded successfully!')

## 1. Mini-BERT Architecture Inspection

We instantiate a lightweight BERT encoder with 2 layers, 4 attention heads, and hidden dimension 64.

In [ ]:
torch.manual_seed(42)
bert = MiniBertModel(vocab_size=500, hidden_dim=64, num_layers=2, num_heads=4, intermediate_dim=128)

# Synthetic batch of 2 sentences, length 6 tokens
input_ids = torch.tensor([
    [101, 24, 78, 92, 102, 0],
    [101, 15, 88, 102, 0, 0]
])
attention_mask = torch.tensor([
    [1, 1, 1, 1, 1, 0],
    [1, 1, 1, 1, 0, 0]
])

hidden, pooled = bert(input_ids, attention_mask=attention_mask)
print(f'Full token sequence representation shape: {hidden.shape}')
print(f'Pooled [CLS] representation shape:         {pooled.shape}')

## 2. Task 1: Sequence Classification (Sentiment Analysis)

We pass the pooled `[CLS]` representation through a dropout layer and linear classifier.

In [ ]:
cls_model = BertForSequenceClassification(bert, num_labels=2)
labels = torch.tensor([1, 0])  # Positive, Negative

out = cls_model(input_ids, attention_mask=attention_mask, labels=labels)
probs = F.softmax(out['logits'], dim=-1)

print(f'Classification Loss: {out["loss"].item():.4f}')
print(f'Predicted Probabilities:\n{probs.detach().numpy()}')

## 3. Task 2: Token Classification (Named Entity Recognition)

We predict entity tags for each token (e.g., O, B-PER, I-PER, B-LOC, I-LOC), using `-100` to ignore padding.

In [ ]:
ner_model = BertForTokenClassification(bert, num_labels=5)
ner_labels = torch.tensor([
    [0, 1, 2, 0, 0, -100],
    [0, 3, 4, 0, -100, -100]
])

ner_out = ner_model(input_ids, attention_mask=attention_mask, labels=ner_labels)
print(f'NER Loss: {ner_out["loss"].item():.4f}')
print(f'Token Logits shape: {ner_out["logits"].shape}')

## 4. Task 3: Extractive Question Answering (SQuAD Span Prediction)

Predicting start and end logits for an answer span within a context document.

In [ ]:
qa_model = BertForQuestionAnswering(bert)
start_pos = torch.tensor([2, 1])
end_pos = torch.tensor([3, 2])

qa_out = qa_model(input_ids, attention_mask=attention_mask, start_positions=start_pos, end_positions=end_pos)
print(f'QA Cross-Entropy Loss: {qa_out["loss"].item():.4f}')

# Extract best span for sequence 0
best_start, best_end = extract_best_answer_span(qa_out['start_logits'][0], qa_out['end_logits'][0])
print(f'Optimal span prediction for sequence 0: tokens [{best_start}..{best_end}]')

## 5. Backpropagation & Optimization Step

Verify that loss gradients flow smoothly through the classification head down into the Transformer embeddings.

In [ ]:
optimizer = torch.optim.AdamW(qa_model.parameters(), lr=1e-3)
optimizer.zero_grad()
loss = qa_out['loss']
loss.backward()
optimizer.step()

emb_grad_norm = bert.embeddings.word_embeddings.weight.grad.norm().item()
print(f'Embedding gradient L2 norm after step: {emb_grad_norm:.6f}')
assert emb_grad_norm > 0, 'Gradients must propagate to embeddings!'